# Notebook 기반 ASR API 실습 개요

이 노트북은 **Whisper 음성인식 모델을 로딩하고**,  
이를 **FastAPI 기반 API 서비스**로 감싸서,  
**노트북 환경에서 직접 실행·제어·모니터링**하는 전체 과정을 다룹니다.

터미널 명령 없이도 버튼 클릭만으로 서버를 실행하고,
외부(`curl`, 다른 프로그램)에서 음성인식 요청을 보내
**JSON 형태의 결과와 로그를 실시간으로 확인**하는 것이 목표입니다.

---

## 전체 구성 요약

이 노트북은 **3개의 핵심 셀**로 구성됩니다.

---

## Cell 1. ASR 모델 로딩 및 음성인식 함수 정의

- Whisper ASR 모델(`openai/whisper-small`)을 **한 번만 로딩**
- GPU/CPU 환경을 자동 감지하여 최적의 dtype 사용
- 음성 파일 1개를 입력 받아
  - 텍스트 변환
  - 모델/디바이스 정보 포함
  - JSON(dict) 형태로 반환하는  
  `transcribe_to_json()` 함수 정의

👉 이후 모든 음성인식 처리는 이 셀에서 로딩한 **ASR 객체를 재사용**

---

## Cell 2. FastAPI 기반 ASR API 정의

- FastAPI 애플리케이션 생성
- API 엔드포인트 제공
  - `/health` : 서버 상태 및 모델 정보 확인
  - `/transcribe` : 오디오 파일 업로드 → 음성인식 → JSON 응답
- 음성인식 요청 시작/종료 시점에 로그 출력
- Uvicorn / FastAPI 로그를
  **노트북 Output 위젯으로 전달하도록 로깅 설정**

👉 “음성인식 기능을 API 서비스로 감싸는 단계”

---

## Cell 3. 서버 제어 UI (Start / Stop / Clear)

- 노트북에서 서버를 직접 제어할 수 있는 버튼 UI 제공
  - **Start** : API 서버 실행
  - **Stop** : 서버 종료
  - **Clear** : 로그 화면 초기화
- 서버를 **별도 스레드**에서 실행하여
  - 서버가 떠 있는 동안에도
  - 다른 셀에서 실습 계속 가능
- 로그 출력 위젯을 통해
  - 서버 상태
  - API 접근 로그
  - ASR 처리 로그
  를 실시간으로 확인

👉 “노트북 안에서 ASR API를 운영하는 컨트롤 패널”

---

## 실습 흐름 한눈에 보기

1. **Cell 1 실행**  
   → ASR 모델 로딩
2. **Cell 2 실행**  
   → API 및 로깅 로직 정의
3. **Cell 3 실행 후 Start 클릭**  
   → 서버 기동
4. 외부 터미널 또는 프로그램에서 `/transcribe` 호출  
   → 음성인식 결과(JSON) + 로그 확인
5. 필요 시 **Stop / Clear** 버튼으로 서버 관리

---

## 이 노트북의 핵심 학습 포인트

- Whisper ASR 모델을 **서비스 형태로 사용하는 구조 이해**
- 노트북 환경에서 서버를 안전하게 띄우는 방법
- 모델 로딩과 서비스 로직을 분리하는 설계 방식
- 실험/수업 환경에 적합한 **경량 API 운영 패턴**

> 이 노트북은  
> **“연구·실습·수업용 ASR API 최소 구현 템플릿”**을 목표로 합니다.


In [1]:
import sys
sys.modules["torchcodec"] = None #원래 없었던거

In [2]:
pip install -U "transformers>=4.40.2" accelerate jiwer soundfile librosa pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install -U "transformers>=4.41" accelerate torch fastapi uvicorn python-multipart soundfile librosa

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip -q install -U transformers accelerate peft safetensors soundfile librosa torch

Note: you may need to restart the kernel to use updated packages.


## Cell 1. Load ASR Model (Whisper) + Define `transcribe_to_json()`

This cell does two things:

1) **Loads a Whisper ASR pipeline once** (GPU if available, otherwise CPU)  
2) Defines **`transcribe_to_json()`**, a helper function that takes an audio file path and returns a **JSON-like `dict`** containing the transcription text and metadata.

### What this cell contains

- **Model selection**
  - `MODEL_ID`: Hugging Face model ID (default: `openai/whisper-small`)
  - `DEFAULT_LANGUAGE`: decoding language (default: `korean`)
  - `DEFAULT_TASK`: decoding task (default: `transcribe`)

- **`load_asr()`**
  - Detects GPU availability with `torch.cuda.is_available()`
  - Sets:
    - `device = 0` for GPU, `device = -1` for CPU
    - `torch_dtype = float16` on GPU, `float32` on CPU
  - Loads:
    - `AutoProcessor` (tokenizer + feature extractor)
    - `AutoModelForSpeechSeq2Seq` (Whisper model)
  - Builds an ASR pipeline via `transformers.pipeline("automatic-speech-recognition", ...)`
  - Returns:
    - `ASR`: the pipeline object
    - `ASR_META`: model/device/dtype metadata

- **`transcribe_to_json()`**
  - Inputs:
    - `audio_path`: audio file path (required)
    - `language`, `task`, `temperature`: decoding options
    - `chunk_s`, `batch_size`: optional chunking for long audio
    - `include_raw`: include raw pipeline output if needed
  - Output:
    - A `dict` containing:
      - `text`: transcription result
      - `model`, `device`, `dtype` (from `ASR_META`)
      - decoding options used

### Notes / Tips

- If you change `MODEL_ID`, re-run this cell to load

In [5]:
import torch, os
from pathlib import Path
from transformers import pipeline as hf_pipeline, AutoProcessor, AutoModelForSpeechSeq2Seq
from peft import PeftModel

# 기존 경로 및 설정 유지
ADAPTER_DIR = "/home/data/bootcamp/user2/outputs_asr/whisper_lora_zeroth/lora_adapter"
MODEL_ID = "openai/whisper-small"
DEFAULT_LANGUAGE = "korean"
DEFAULT_TASK = "transcribe"

def load_asr(model_id: str = MODEL_ID, lora_adapter_dir: str | None = None, merge_lora: bool = False):
    if torch.cuda.is_available():
        device = 0
        torch_dtype = torch.float16
        device_name = "cuda"
    else:
        device = -1
        torch_dtype = torch.float32
        device_name = "cpu"

    processor = AutoProcessor.from_pretrained(model_id)
    # 14B 모델과의 VRAM 경합을 피하기 위해 low_cpu_mem_usage 옵션 권장
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, 
        torch_dtype=torch_dtype, 
        low_cpu_mem_usage=True
    )
    
    if torch.cuda.is_available():
        model = model.to("cuda")
        
    if lora_adapter_dir:
        model = PeftModel.from_pretrained(model, lora_adapter_dir)
        if merge_lora:
            model = model.merge_and_unload()
    model.eval()

    # 파이프라인 생성
    asr = hf_pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        device=device,
    )
    meta = {
        "model": model_id, 
        "lora_adapter_dir": lora_adapter_dir or "", 
        "device": device_name, 
        "dtype": str(torch_dtype).replace("torch.", "")
    }
    return asr, meta

def transcribe_to_json(
    audio_path: str,
    language: str = DEFAULT_LANGUAGE,
    task: str = DEFAULT_TASK,
    temperature: float = 0.0,
    chunk_s: int = 30,  # 기본값을 30으로 수정하여 안정성 확보
    batch_size: int = 8,
    include_raw: bool = False,
):
    p = Path(audio_path).expanduser().resolve()
    if not p.exists():
        raise FileNotFoundError(f"Audio not found: {p}")

    # generate_kwargs 설정
    generate_kwargs = {
        "language": language, 
        "task": task, 
        "temperature": temperature
    }
    
    # [핵심 수정] KeyError: 'num_frames' 방지를 위한 강제 옵션 적용
    call_kwargs = {
        "generate_kwargs": generate_kwargs,
        "chunk_length_s": chunk_s if chunk_s > 0 else 30, # 0일 경우에도 30초로 강제
        "batch_size": batch_size,
        "return_timestamps": True # 타임스탬프 반환 시 내부 로직이 더 안정적으로 작동함
    }

    try:
        # ASR 실행
        out = ASR(str(p), **call_kwargs)
    except Exception as e:
        # 시스템 중단 방지를 위한 에러 핸들링
        print(f"[ASR Error] {type(e).__name__}: {e}")
        out = {"text": "[음성 인식 오류: 짧은 무음이나 데이터 손상]"}

    text = (out.get("text") if isinstance(out, dict) else str(out)) or ""
    text = text.strip()

    payload = {
        "audio_path": str(p),
        "text": text,
        "language": language,
        "task": task,
        "temperature": temperature,
        "chunk_length_s": call_kwargs["chunk_length_s"],
        **ASR_META,
    }
    if include_raw:
        payload["raw"] = out
    return payload

# 모델 로드 및 메타 정보 확인
ASR, ASR_META = load_asr(MODEL_ID, ADAPTER_DIR)
print(f"ASR Model Loaded on {ASR_META['device']}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

ASR Model Loaded on cuda


## Cell 2. FastAPI 기반 ASR API 정의 + 로그 처리 설정

이 셀은 **노트북에서 ASR 기능을 API 형태로 제공하기 위한 핵심 로직**을 정의합니다.  
앞선 Cell 1에서 로딩한 **ASR 모델과 `transcribe_to_json()` 함수**를 그대로 재사용합니다.

---

### 이 셀의 역할 요약

1) **FastAPI 애플리케이션 생성**
2) **ASR API 엔드포인트 정의**
3) **Uvicorn / FastAPI 로그를 노트북 위젯으로 전달하기 위한 로깅 설정**

---

## 1) FastAPI 애플리케이션 생성

```python
app = FastAPI(title="Notebook ASR API")

In [6]:
import nest_asyncio
nest_asyncio.apply()

import os, tempfile, time, logging
from pathlib import Path
from fastapi import FastAPI, UploadFile, File, Form

# ---- app ----
app = FastAPI(title="Notebook ASR API")

# ---- widget logger hook (out 위젯은 다음 셀에서 생성하므로, hook 함수만 여기서 정의) ----
WIDGET_HANDLER_NAME = "widget_out_handler"

class OutputWidgetHandler(logging.Handler):
    def __init__(self, out_widget):
        super().__init__()
        self.out_widget = out_widget
        self.name = WIDGET_HANDLER_NAME

    def emit(self, record):
        try:
            msg = self.format(record)
            self.out_widget.append_stdout(msg + "\n")
        except Exception:
            pass

def hook_logs_to_widget(out_widget):
    """
    uvicorn/fastapi 로그를 out_widget으로 보냄.
    - handlers 싹 비우고 1개만 사용
    - propagate False로 중복 방지
    """
    handler = OutputWidgetHandler(out_widget)
    handler.setLevel(logging.INFO)
    handler.setFormatter(logging.Formatter("[%(levelname)s] %(name)s: %(message)s"))

    for lname in ["fastapi", "uvicorn", "uvicorn.error", "uvicorn.access"]:
        lg = logging.getLogger(lname)
        lg.setLevel(logging.INFO)
        lg.propagate = False
        lg.handlers = []          # ✅ 중복 방지
        lg.addHandler(handler)

log = logging.getLogger("fastapi")

@app.get("/health")
def health():
    log.info("[HEALTH] ok")
    return {"ok": True, **ASR_META}

@app.post("/transcribe")
async def transcribe(
    audio: UploadFile = File(...),
    language: str = Form(DEFAULT_LANGUAGE),
    task: str = Form(DEFAULT_TASK),
    temperature: float = Form(0.0),
    chunk_s: int = Form(0),
    batch_size: int = Form(8),
    include_raw: bool = Form(False),
):
    log.info(f"[ASR] request: file={audio.filename} lang={language} task={task} chunk_s={chunk_s}")

    suffix = Path(audio.filename or "").suffix or ".wav"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as f:
        tmp_path = f.name
        f.write(await audio.read())

    t0 = time.time()
    try:
        result = transcribe_to_json(
            tmp_path,
            language=language,
            task=task,
            temperature=temperature,
            chunk_s=chunk_s,
            batch_size=batch_size,
            include_raw=include_raw,
        )
        dt_ms = int((time.time() - t0) * 1000)
        log.info(f"[ASR] done: ms={dt_ms} text={(result.get('text',''))}")

        # 임시파일 경로 노출 방지
        result["audio_path"] = audio.filename or "uploaded_audio"
        return result
    except Exception as e:
        log.exception(f"[ASR] error: {type(e).__name__}: {e}")
        raise
    finally:
        try:
            os.remove(tmp_path)
        except OSError:
            pass

## Cell 3. 서버 제어 UI (Start / Stop / Clear) + 로그 출력 위젯

이 셀은 노트북에서 FastAPI 서버를 **버튼으로 쉽게 제어**하기 위한 UI를 제공합니다.  
수업 환경에서는 터미널 명령 없이도 **서버 실행/종료/로그 확인**을 할 수 있게 만드는 것이 목적입니다.

---

### 이 셀의 역할 요약

1) **서버 제어 버튼 UI 구성**
   - Start: 서버 실행
   - Stop: 서버 종료
   - Clear: 로그 화면 초기화

2) **로그 출력 영역(Output 위젯) 제공**
   - Uvicorn 시스템 로그
   - API 접근 로그(uvicorn.access)
   - ASR 처리 로그(fastapi logger)

3) **서버를 별도 스레드에서 실행**
   - 노트북 셀이 서버 실행에 묶이지 않도록 함  
   - 서버 실행 중에도 다른 셀에서 실습을 계속 진행 가능

---

## 1) UI 구성 요소

- `out` : 로그를 표시하는 Output 위젯(스크롤 가능)
- `host_w`, `port_w` : 서버 주소/포트를 입력하는 위젯
- `start_btn`, `stop_btn`, `clear_btn` : 제어 버튼
- `status` : 현재 서버 상태 표시 (stopped / running / stopping 등)

---

## 2) Start 버튼 동작 (서버 실행)

Start 버튼을 누르면 다음 순서로 동작합니다.

1. 현재 서버가 이미 실행 중인지 확인  
2. `hook_logs_to_widget(out)` 실행  
   - **Uvicorn / FastAPI 로그를 Output 위젯으로 연결**
3. `uvicorn.Config(...)` 생성
   - `access_log=True` : 요청 접근 로그 활성화
   - `log_config=None` : uvicorn이 로깅 설정을 다시 덮어쓰는 것을 최소화
4. `uvicorn.Server(config)` 생성
5. **별도 스레드(thread)** 에서 `asyncio.run(server.serve())` 실행
6. 상태를 `running`으로 변경하고 안내 로그 출력

> 핵심: 서버를 스레드로 실행하기 때문에  
> **노트북이 멈추지 않고 다른 셀을 계속 실행할 수 있습니다.**

---

## 3) Stop 버튼 동작 (서버 종료)

Stop 버튼을 누르면:

1. 실행 중인지 확인
2. `_server.should_exit = True`로 종료 신호 전달
3. `thread.join(timeout=...)`으로 종료 대기
4. 종료되면 상태를 `stopped`로 변경
5. 서버 관련 객체를 초기화하여 다음 실행에 대비

---

## 4) Clear 버튼 동작 (로그 초기화)

Clear 버튼을 누르면:

- `out.clear_output()`으로 로그 화면을 비움
- 바로 아래에 `[INFO] Logs cleared.` 같은 안내 로그를 다시 출력

수업 중 로그가 많이 쌓였을 때 **화면을 정리하는 용도**로 유용합니다.

---

## 수업 포인트 정리

- 이 셀은 **서버를 버튼으로 제어하는 운영 UI**
- 서버 실행 중에도 실습을 계속할 수 있게 **스레드 기반 실행**을 사용
- 로그를 한 화면에서 확인할 수 있어:
  - 요청이 들어왔는지
  - 처리 시간이 얼마나 걸렸는지
  - 서버가 정상 동작 중인지  
  를 빠르게 점검할 수 있음

> 즉, 이 셀은  
> **“노트북에서 ASR API 서버를 쉽게 운영하는 컨트롤 패널”** 입니다.


In [7]:
import threading, asyncio, time
import ipywidgets as widgets
from IPython.display import display
import uvicorn

# ---- widgets ----
out = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px", height="280px", overflow="auto"))
port_w = widgets.IntText(value=8002, description="Port:", layout=widgets.Layout(width="200px"))
host_w = widgets.Text(value="0.0.0.0", description="Host:", layout=widgets.Layout(width="260px"))

start_btn = widgets.Button(description="Start", button_style="success")
stop_btn  = widgets.Button(description="Stop", button_style="danger")
clear_btn = widgets.Button(description="Clear", button_style="")  # ✅ 추가

status = widgets.HTML(value="<b>Status:</b> stopped")

display(widgets.HBox([host_w, port_w, start_btn, stop_btn, clear_btn, status]), out)

def ui_log(msg: str):
    out.append_stdout(msg + "\n")

# ---- server state ----
_server = None
_thread = None

def start_server(_=None):
    global _server, _thread
    if _thread and _thread.is_alive():
        ui_log("[INFO] Server already running.")
        status.value = "<b>Status:</b> running"
        return

    # ✅ 로그를 위젯으로 연결 (중복 없이 1회)
    hook_logs_to_widget(out)

    host = host_w.value.strip()
    port = int(port_w.value)

    config = uvicorn.Config(
        app,
        host=host,
        port=port,
        log_level="info",
        access_log=True,
        log_config=None,
    )
    _server = uvicorn.Server(config)

    def runner():
        try:
            ui_log(f"[INFO] Starting server on http://{host}:{port}")
            asyncio.run(_server.serve())
            ui_log("[INFO] Server exited.")
        except Exception as e:
            ui_log(f"[ERROR] {type(e).__name__}: {e}")

    _thread = threading.Thread(target=runner, daemon=True)
    _thread.start()

    time.sleep(0.3)
    status.value = "<b>Status:</b> running"
    ui_log("[INFO] Ready. Try: GET /health or POST /transcribe")

def stop_server(_=None):
    global _server, _thread
    if not _thread or not _thread.is_alive():
        ui_log("[INFO] Server is not running.")
        status.value = "<b>Status:</b> stopped"
        return

    ui_log("[INFO] Stopping server...")
    _server.should_exit = True
    _thread.join(timeout=2.0)

    if _thread.is_alive():
        ui_log("[WARN] Server thread still alive (may stop shortly).")
        status.value = "<b>Status:</b> stopping..."
    else:
        ui_log("[INFO] Server stopped.")
        status.value = "<b>Status:</b> stopped"

    _server = None
    _thread = None

def clear_logs(_=None):
    # ✅ Output 위젯 내용 초기화
    out.clear_output()
    ui_log("[INFO] Logs cleared.")

start_btn.on_click(start_server)
stop_btn.on_click(stop_server)
clear_btn.on_click(clear_logs)

ui_log("[INFO] UI ready. Click Start.")

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…